# Credit Manager - Database Initialization Test
Este notebook inicializa la base de datos SQLite y verifica que todas las tablas mapeadas en SQLAlchemy se hayan creado correctamente a partir de nuestro módulo `src/database`.

In [ ]:
import sys
import os

from importlib import reload

# Asegurar que el path apunte a la raíz para encontrcar el paquete 'src'
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

In [ ]:
"""
Notebook Cell: Database Reset & Modular Seeding
Description: Drops tables, recreates them, and calls the external seeding module.
Author: Juan Martín Carini
Date: 2026-05-11
"""


import src.database  # noqa: E402
import src.database.seed_geography  # noqa: E402

def reset_and_seed():
    print("Iniciando reset de base de datos...")
    
    # 1. Limpieza total
    src.database.Base.metadata.drop_all(bind=src.database.engine)
    print("✅ Todas las tablas han sido eliminadas.")
    
    # 2. Reconstrucción del esquema
    src.database.Base.metadata.create_all(bind=src.database.engine)
    print("✅ Estructura de tablas recreada correctamente.")

    # 3. Carga de datos maestros usando el nuevo módulo
    db = src.database.SessionLocal()
    try:
        print("Poblando tablas geográficas...")
        src.database.seed_geography.seed_provincias(db)
    except Exception as e:
        db.rollback()
        print(f"❌ Error durante el seeding: {e}")
    finally:
        db.close()

if __name__ == "__main__":
    reset_and_seed()

In [ ]:
import src.etl.csv_importer as importer
reload(importer)

path_personas = "../data/PERSONAS.CSV"
path_prestamos = "../data/PRESTAMOS.CSV"
path_cuotas = "../data/CUOTAS.CSV"


NewFolder = importer.PortfolioImporter()
NewFolder.create_portfolio("Folder Test", "2026/03/10", 0.40, 30713257880, "Mentiritas S.A.", recurso=True, iva=False)
NewFolder.read_csv(path_personas, path_prestamos, path_cuotas)
NewFolder.validation()
NewFolder.check_warnings()
NewFolder.save_portfolio()

In [ ]:
from src.reports import saldos

df = saldos(con_saldo=False, propias=True, agrupar=True, socios=True, vencimientos=True)
df[["Capital", "Interés", "IVA", "Total"]] = df[["Capital", "Interés", "IVA", "Total"]].map("$ {:,.2f}".format)

df

In [ ]:
"""
=============================================================================
Snippet: Fetch Purchased Installments by Commercial Partner CUIT
Description: Executes an optimized SQL query to retrieve all commercialized
             installments purchased from a specific partner using parameter binding.
=============================================================================
"""

import pandas as pd  # noqa: E402
from sqlalchemy import text  # noqa: E402, F401
from src.database import engine  # noqa: E402
import src.logic.collections as collections  # noqa: E402

reload(collections)

NewColl = collections.CollectionManager()

# 1. Definición del identificador del socio comercial (Tax ID)
cuil_socio = 30713257880

a_cancelar = 214907675.40-1000
NewColl.process_resource("2026/03/28", a_cancelar+1000, 30713257880, "PROVEEDOR_CUIT")

In [ ]:
from src.reports import saldos
from src.database import engine  # noqa: F811

df = saldos("2026/05/19", con_saldo=False, propias=True, agrupar=True, socios=True, vencimientos=True)

display(pd.read_sql("anticipos_socios", engine))

df.map("$ {:,.2f}".format)